# Run TMB_MPNN sequence design

This notebook runs TMB_MPNN on backbone PDBs generated from RFD2.

Make sure you have:
- downloaded the MPNN checkpoint
- downloaded the Apptainer container
- set environment variables:
  - `MPNN_CKPT_PATH`
  - `APPTAINER_PATH`


In [ ]:
import os
import subprocess
from pathlib import Path
import shlex

## Load environment variables

In [ ]:
MPNN_CKPT_PATH = os.environ.get("MPNN_CKPT_PATH")
APPTAINER_PATH = os.environ.get("APPTAINER_PATH")
TMB_PIPELINE_DIR = Path(os.environ.get("TMB_PIPELINE_DIR", os.getcwd()))

assert MPNN_CKPT_PATH, "Set MPNN_CKPT_PATH"
assert APPTAINER_PATH, "Set APPTAINER_PATH"

MPNN_CKPT_PATH = Path(MPNN_CKPT_PATH)
APPTAINER_PATH = Path(APPTAINER_PATH)

print("Checkpoint:", MPNN_CKPT_PATH)
print("Container:", APPTAINER_PATH)

## User settings

In [ ]:
PDB_DIR = TMB_PIPELINE_DIR / "notebook_runs" / "backbones"
OUT_FOLDER = TMB_PIPELINE_DIR / "notebook_runs" / "mpnn_outputs"

MAKE_JSON_SCRIPT = TMB_PIPELINE_DIR / "scripts" / "make_pdb_multi_surf_cond_json.py"
MPNN_CODE_DIR = TMB_PIPELINE_DIR / "mpnn"

MODEL_CONFIG = "mpnn/configs/big_tmbb_80p_surf_condition.py"

BATCH_SIZE = 5
NUM_BATCHES = 2
SERTHRE_THRESHOLD = 0.25
SERTHRE_BIAS = -0.5
PACK_SIDE_CHAINS = 1

OUT_FOLDER.mkdir(parents=True, exist_ok=True)

print("PDB_DIR:", PDB_DIR)
print("OUT_FOLDER:", OUT_FOLDER)

## Helper function

In [ ]:
def run_command(cmd, cwd=None):
    print("\nRunning:\n", cmd)
    result = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError("Command failed")
    return result

## Generate JSON inputs

In [ ]:
os.chdir(PDB_DIR)
cmd = f"python {MAKE_JSON_SCRIPT} 100.0 0.25 *.pdb"
run_command(cmd)

## Run MPNN

In [ ]:
inner_cmd = f"""
cd {TMB_PIPELINE_DIR}

python {MPNN_CODE_DIR / 'run.py'} \
    --checkpoint {MPNN_CKPT_PATH} \
    --model_type {MPNN_CODE_DIR / MODEL_CONFIG} \
    --pdb_path_multi ./pdbs_multi_json.jsonl \
    --out_folder {OUT_FOLDER} \
    --batch_size {BATCH_SIZE} \
    --number_of_batches {NUM_BATCHES} \
    --pack_side_chains {PACK_SIDE_CHAINS} \
    --threonine_serine_threshold {SERTHRE_THRESHOLD} \
    --serthre_bias {SERTHRE_BIAS} \
    --tmbb_surface_condition_multi ./intf_buried.jsonl
""".strip()

apptainer_cmd = f"apptainer run --nv {APPTAINER_PATH} bash -lc '{inner_cmd}'"

run_command(apptainer_cmd, cwd=PDB_DIR)